# P8 – Séance de travail : régimes actions-taux

Ce notebook remplace le terminal. **Une seule cellule à modifier : les paramètres (section 1)**, puis
`Exécuter tout`. Chaque séance laisse trois traces :

* une ligne dans `journal/journal.md` (si `SAUVEGARDER = True`) ;
* les figures et tableaux dans `outputs/AAAA-MM-JJ/` ;
* une copie HTML du notebook (dernière cellule, après `Ctrl+S`).

Rappel de la méthode (note théorique, sections 3 et 4) : chaque fin de mois, un VAR quotidien sur
les taux 2/5/10 ans et les actions est identifié par restrictions de signe (Cieslak et Pang) sur les
500 derniers jours ; l'indicateur **HB** est la part du risque actions des 3 derniers mois qui vient des
chocs monétaires et de prime commune, ceux qui font baisser actions et obligations ensemble.
Régime **cash** si HB ≥ 50 % (P8-HB) ou si le bêta prévu des obligations sur les actions est positif (P8-bêta).

## 1. Paramètres

In [ ]:
MARCHE = "US"          # "US" (Treasuries + marché américain) ou "EA" (courbe allemande + DAX)
TELECHARGER = False    # True : retélécharge les données publiques (une fois par semaine suffit)
DEMO = False           # True : données simulées, sans internet (pour vérifier l'installation)
N_ROTATIONS = 1000     # rotations admissibles par date ; 300 pour un essai rapide
RECALCULER = False     # True : ignore le cache de l'indicateur et recalcule tout
AVEC_HMM = True        # False : saute le concurrent HMM et son test de stabilité
ROBUSTESSE_P8 = False  # True : recalcule HB pour 5 jeux de fenêtres (long la première fois, puis en cache)
SAUVEGARDER = True     # écrit le journal et les tableaux dans outputs/AAAA-MM-JJ
NOTE = ""              # remarque libre ajoutée au journal

## 2. Préparation

In [ ]:
%matplotlib inline
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from p8regimes.config import Params, get_paths, SHOCK_LABELS
from p8regimes import data, dataset, indicator, signals, allocation, backtest, report
from p8regimes import evaluation as ev
from p8regimes.journal import write_journal, export_notebook_html
from p8regimes.pipeline import Session, demo_panel, save_tables
from p8regimes.structural import LEVELS

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

PATHS = get_paths()
P = Params(n_accept=N_ROTATIONS)
SORTIES = PATHS.outputs / datetime.now().strftime("%Y-%m-%d")
print("dépôt :", PATHS.root)
print("sorties :", SORTIES)
print("clé du cache de l'indicateur :", P.indicator_key())

## 3. Téléchargement des données

In [ ]:
if TELECHARGER and not DEMO:
    data.fetch(MARCHE, PATHS)
data.data_status(PATHS)

## 4. Vérification des données

Contrôles : dates de début et de fin, trous dans le calendrier, volatilités annuelles (actions vers 15-20 %,
poche taux 7 ans vers 5-7 %), corrélation actions-taux sur tout l'échantillon.

In [ ]:
if DEMO:
    panel, choc_vrais = demo_panel(P)
else:
    panel, choc_vrais = dataset.build_panel(MARCHE, P, PATHS), None
for n in panel.notes:
    print("-", n)
display(panel.summary())
report.fig_data(panel, SORTIES if SAUVEGARDER else None);

### Contrôle de la poche taux contre un fonds négociable

La poche taux est un zéro-coupon 7 ans reconstruit à partir de la courbe estimée : un titre fictif.
On la compare à l'ETF iShares 7-10 Year Treasury (**IEF**), en rendement total, depuis 2002 (États-Unis).
Repères : corrélation mensuelle ≥ 0,95 et bêta de IEF sur la poche entre 0,8 et 1,2. Un écart moyen
légèrement négatif est normal : frais du fonds (0,15 %/an), coupons, duration de l'indice un peu différente de 7.

In [ ]:
from p8regimes.checks import compare_bond_pocket
controle = None
if panel.bond_benchmark is not None:
    controle = compare_bond_pocket(panel.daily, panel.bond_benchmark, panel.bond_benchmark_name)
    display(controle["stats"].to_frame())
    display(Markdown(controle["verdict"]))
    report.fig_pocket_check(controle, panel.market, SORTIES if SAUVEGARDER else None);
elif MARCHE == "US":
    print("IEF absent : relancer avec TELECHARGER = True (ou déposer ief.csv dans data/raw/manual).")
else:
    print("Pas encore de fonds de référence pour la zone euro : contrôle disponible aux États-Unis.")

## 5. Indicateur en temps réel

La première exécution calcule toutes les fins de mois (quelques minutes, avec une barre d'avancement) ;
ensuite, seul le nouveau mois est calculé (cache dans `data/processed/`). La colonne `level` indique le jeu de
restrictions utilisé : 0 = Cieslak-Pang complet ; au-delà, restrictions allégées sur le 2 ans
(taux courts bloqués près de zéro).

In [ ]:
ind = indicator.run_indicator(panel, P, PATHS, use_cache=not RECALCULER, progress=True)
rs = indicator.realized_stats(panel.daily, ind.index, P.recent)
cols = ["hb", "hb_p16", "hb_p84", "level", "beta_now", "beta_fcst", "s_g", "s_m", "s_pp", "s_pc", "acceptance"]
display(ind[[c for c in cols if c in ind]].tail(12))
niveaux = ind["level"].value_counts().sort_index()
display(pd.DataFrame({"fins de mois": niveaux, "restrictions": [LEVELS.get(int(k), "aucune admissible") for k in niveaux.index]}))
if choc_vrais is not None:
    from p8regimes.simulate import true_hb
    vrai = true_hb(choc_vrais.to_numpy(), panel.daily.index, ind.index, P.recent)
    print(f"DÉMO : corrélation entre HB estimé et vraie part = {ind['hb'].corr(vrai):.3f}")

In [ ]:
report.fig_hb(ind, rs, panel.market, P.hb_threshold, SORTIES if SAUVEGARDER else None);
report.fig_shares(ind, panel.market, SORTIES if SAUVEGARDER else None);

## 6. Le régime du mois

In [ ]:
sig = signals.build_signals(panel, ind, rs, P, with_hmm=AVEC_HMM)
risk = allocation.risk_inputs(panel.daily, ind.index, P)
last = ind.dropna(subset=["hb"]).iloc[-1]; d = last.name
parts = {SHOCK_LABELS[k]: last[f"s_{k}"] for k in ("g", "pp", "m", "pc")}
dominant = max(parts, key=parts.get)
regime = "CASH (réduire les deux poches)" if sig.loc[d, "C_hb"] == 1 else "COUVERTURE (garder le 60/40)"
bf = last.get("beta_fcst", np.nan)
provisoire = dataset.month_in_progress(d, panel.daily.index[-1])
quand = f"{d.date()} (mois en cours : lecture provisoire)" if provisoire else f"{d.date()}"
txt = f"""**{panel.market}, {quand}** : {100 * last['hb']:.0f} % du risque actions des trois derniers mois vient des chocs
qui cassent la couverture (bande d'identification {100 * last['hb_p16']:.0f}-{100 * last['hb_p84']:.0f} %).
Choc dominant : **{dominant}** ({100 * parts[dominant]:.0f} %). Régime P8-HB : **{regime}**.
Bêta prévu de la poche taux sur les actions pour le trimestre suivant : {bf:+.2f}
({'cash' if sig.loc[d, 'C_beta'] == 1 else 'couverture'} pour P8-bêta).
Corrélation réalisée sur 63 jours : {rs.loc[d, 'corr']:+.2f} ; facteur de volatilité k = {risk.loc[d, 'k']:.2f}."""
display(Markdown(txt))
display(pd.DataFrame(parts, index=["part de la variance des actions"]).T)

## 7. Lecture des épisodes (hypothèses de la note, section 5.5)

In [ ]:
ep = ev.episodes_table(ind, rs, MARCHE if not DEMO else "XX")
display(ep if len(ep) else Markdown("_Pas d'épisode de la liste dans l'échantillon (normal en mode démo)._"))

## 8. Backtest : la course de chevaux

Même moteur pour tous, seul le signal change. Décision à la fin du mois, appliquée le mois suivant,
coûts de 2 pb par unité de rotation. Variante A : 60/40 réduit au risque de référence en régime cash ;
variante B : poche taux sortie, actions au risque de référence.

In [ ]:
mon = dataset.monthly_returns(panel.daily)
strategies = backtest.default_strategies(sig)
res = backtest.run_backtest(mon, sig, risk, strategies, P)
perf = {"tout l'échantillon": ev.perf_table(res)}
if not DEMO:
    from p8regimes.config import MARKETS
    for a, b in MARKETS[MARCHE].subperiods:
        if len(res["60/40"].loc[a:b]) >= 24:
            perf[f"{a[:4]}-{b[:4] if b < '2100' else 'fin'}"] = ev.perf_table(res, a, b)
print(f"backtest : {res['60/40'].index[0].date()} -> {res['60/40'].index[-1].date()}, {len(res['60/40'])} mois")
for k, t in perf.items():
    display(Markdown(f"**Période : {k}**"))
    display(t[["rendement (%/an)", "volatilité (%/an)", "Sharpe", "perte max (%)", "CVaR 5 % (%/mois)", "Calmar",
               "rotation (/an)", "temps en cash (%)", "pires mois actions (%/mois)"]].round(2))
report.fig_performance(res, panel.market, out_dir=SORTIES if SAUVEGARDER else None);

## 9. Tests statistiques et décomposition face à la volatilité cible

Écarts de Sharpe par bootstrap par blocs circulaires (blocs de 6 mois). La proposition 4 de la note dit que
P8-A ne diffère de la volatilité cible que les mois « couverture, volatilité haute » : le tableau du bas le vérifie.

In [ ]:
tests = ev.tests_table(res, params=P)
keep = [i for i in tests.index if i[0].startswith(("P8", "CORR", "HMM", "MACRO", "ORACLE"))]
display(tests.loc[keep].round(3))
vt = pd.concat([ev.vt_decomposition(res, n) for n in ("P8-HB-A", "P8-BETA-A")], axis=1)
display(vt.round(3))

## 9 bis. Robustesse et discipline

Quatre contrôles pour se méfier de ses propres résultats.

1. **Sharpe déflaté** (Bailey et López de Prado, 2014). On a essayé une douzaine de stratégies : la meilleure
   paraît bonne en partie par chance. Le DSR est la probabilité que le vrai ratio dépasse ce que la chance seule
   donnerait au meilleur de N essais, en tenant compte de la longueur de l'échantillon, de l'asymétrie et des
   queues épaisses. Le test le plus exigeant porte sur le **rendement actif** (stratégie moins 60/40, ou moins
   volatilité cible). N effectif corrige le fait que les stratégies se ressemblent. Repère : DSR > 0,95.
2. **Bascules de régime et coûts** : fréquence des bascules, allers-retours en moins de deux mois, part de la
   rotation due aux bascules, et coût par unité de rotation au-delà duquel l'avantage disparaît.
3. **Stabilité du HMM** : nombre d'états, fenêtre d'estimation, initialisation ; et, à chaque ré-estimation, part
   des régimes passés inchangés et ré-estimations qui changent à elles seules le régime du mois.
4. **Stabilité de P8** (si `ROBUSTESSE_P8 = True`) : régimes recalculés avec d'autres fenêtres (W, h).

In [ ]:
from p8regimes import robustness as rb
robust = {"dsr": rb.deflated_sharpe_table(res), "dsr_6040": rb.deflated_sharpe_table(res, "60/40"),
          "dsr_vc": rb.deflated_sharpe_table(res, "VC")}
for k, titre in (("dsr", "Sharpe de chaque stratégie (contre le cash)"),
                 ("dsr_6040", "Rendement actif contre le 60/40"), ("dsr_vc", "Rendement actif contre la volatilité cible")):
    a = robust[k].attrs
    display(Markdown(f"**{titre}** : N = {a['N']} essais, N effectif = {a['N_eff']:.1f}, {a['mois']} mois"))
    display(robust[k].round(3))

In [ ]:
robust["switches"] = rb.switch_stats(res, P.cost_bp)
robust["costs"] = rb.cost_grid(mon, sig, risk, strategies, P)
robust["break_even"] = rb.break_even_costs(robust["costs"])
display(Markdown("**Bascules de régime**")); display(robust["switches"].round(2))
display(Markdown("**Sharpe selon le coût par unité de rotation**")); display(robust["costs"].round(3))
display(Markdown("**Coût au-delà duquel l'avantage disparaît**")); display(robust["break_even"])

In [ ]:
if AVEC_HMM:
    robust["hmm"] = rb.hmm_stability(mon, risk, P, ind.index)
    display(Markdown("**Stabilité du HMM selon la configuration**")); display(robust["hmm"]["table"].round(2))
    display(Markdown("**Ré-estimations successives (configuration de base)**")); display(robust["hmm"]["stability"].round(3))
if ROBUSTESSE_P8:
    robust["p8_windows"] = rb.indicator_sensitivity(panel, P, mon, risk, PATHS)
    display(Markdown(f"**Stabilité de P8 selon les fenêtres** (300 rotations par date, depuis {robust['p8_windows'].attrs['début commun']})"))
    display(robust["p8_windows"].round(3))

## 10. Critère n° 3 : l'indicateur prévoit-il la corrélation du trimestre suivant ?

Modèle 1 : corrélation du trimestre écoulé. Modèle 2 : on ajoute HB, ou le bêta prévu. Prévisions hors
échantillon en fenêtre croissante ; test de Clark et West (unilatéral). C'est le test central du projet.

Le deuxième tableau refait le test par sous-période (contrôle de robustesse, non pré-enregistré) : les
prévisions sont les mêmes, seule la période d'évaluation change. Aux États-Unis, la ligne « 2000-fin » dit
si le résultat tient une fois passée la grande bascule de régime de 1998-2000.

In [ ]:
fc = ev.forecast_test(ind, rs, P)
display(fc)
from p8regimes.config import MARKETS
fc_periodes = ev.forecast_by_period(ind, rs, P, None if DEMO else MARKETS[MARCHE].forecast_periods)
display(Markdown("**Par sous-période** (robustesse) :")); display(fc_periodes.round(3))
fc_beta = ev.forecast_test(ind, rs, P, target="beta_next")
display(Markdown("_Même test sur le bêta réalisé :_")); display(fc_beta)
report.fig_scatter(ind, rs, panel.market, SORTIES if SAUVEGARDER else None);

## 11. Critères de réussite (fixés dans la note de cadrage)

In [ ]:
crit = ev.success_criteria(perf["tout l'échantillon"], tests, fc)
display(crit)
print("Critère 3 (Clark-West à 10 %) :", crit.attrs.get("critère 3", {}))

## 12. Journal et tableaux

In [ ]:
session = Session(panel.market, P, panel, ind, rs, sig, risk, res, perf=perf, tests=tests, vt=vt,
                  forecast=fc, forecast_periods=fc_periodes, episodes=ep, criteria=crit,
                  pocket_check=controle, robust=robust)
if SAUVEGARDER:
    j = write_journal(session, NOTE, PATHS)
    fichiers = save_tables(session, SORTIES)
    print("journal :", j)
    print(len(fichiers), "tableaux dans", SORTIES)
else:
    print("SAUVEGARDER = False : rien n'est écrit")
FIN_SEANCE = datetime.now()

## 13. Copie HTML de la séance

La copie part du notebook **enregistré sur le disque**. Pendant « Exécuter tout », cette cellule refuse donc la
copie (le notebook n'est pas encore enregistré) : c'est normal. Fais `Ctrl+S`, puis relance uniquement cette cellule.

In [ ]:
from p8regimes.journal import NotebookNotSaved
NOTEBOOK = PATHS.root / "notebooks" / "02_seance_de_travail.ipynb"
if SAUVEGARDER:
    try:
        print("copie HTML :", export_notebook_html(NOTEBOOK, PATHS, after=FIN_SEANCE,
                                                   market=None if DEMO else MARCHE))
    except NotebookNotSaved as e:
        print("copie HTML non faite :", e)
    except Exception as e:
        print("copie HTML impossible :", e)